# 实验6：MobileNet模型构建及代码实现

## 实验目的

1. 理解轻量级网络的设计动机
2. 掌握深度可分离卷积（Depthwise Separable Convolution）的原理
3. 理解MobileNet的网络结构和宽度乘子（alpha）的作用
4. 使用TensorFlow/Keras实现MobileNet
5. 对比标准卷积与深度可分离卷积的参数量

## 实验原理

### 轻量级网络的需求

传统CNN（如VGG-16约138M参数、15.5G FLOPs）难以部署在移动设备和嵌入式设备上。**FLOPs（Floating Point Operations，浮点运算次数）**是衡量模型计算开销的核心指标——FLOPs越低，推理速度越快、能耗越低。MobileNet（α=1.0）仅需约569M FLOPs，相比VGG-16减少约**27倍**。

### 标准卷积 vs 深度可分离卷积——分步理解

深度可分离卷积的核心思想是将标准卷积**拆分为两步**：

**第一步：标准卷积（对比理解）**
- 一个卷积核 (K×K×Cin) 同时处理所有通道 → 空间滤波和通道混合**同时进行** → 计算量大
- 参数量：K² × Cin × Cout

**第二步：逐深度卷积（Depthwise）——只做空间滤波**
- 每个通道各自拥有一个 K×K 卷积核，**独立**进行空间卷积
- 优点：计算量极低；缺点：没有跨通道信息交互
- 参数量：K² × Cin

**第三步：逐点卷积（Pointwise, 1×1卷积）——只做通道混合**
- 在每个空间位置上对所有通道做线性组合，**完成跨通道信息融合**
- 参数量：Cin × Cout

**FLOPs对比**：
| 类型 | FLOPs公式 |
|------|----------|
| 标准卷积 | K² × Cin × Cout × H × W |
| 深度可分离卷积 | K² × Cin × H × W + Cin × Cout × H × W |
| **压缩比** | **1/Cout + 1/K²（3×3卷积约减少为1/9）** |

以3×3卷积、256→256通道为例：标准卷积 589,824 参数 vs 深度可分离 67,840 参数（**减少约8.7倍**）

### MobileNet深度可分离卷积块结构

```
输入 → DepthwiseConv2D(3×3) → BN → ReLU6 → Conv2D(1×1) → BN → ReLU6 → 输出
```

**ReLU6激活函数**：ReLU6(x) = min(max(0, x), 6)，将输出截断在[0, 6]区间。相比标准ReLU（输出范围[0, +∞)），ReLU6的有界输出对移动端8位定点量化（INT8）非常友好——有限的数值范围使量化后精度损失更小，是MobileNet适合移动部署的关键设计之一。

### 宽度乘子α

宽度乘子α用于控制网络宽度，通过缩放每层的通道数来平衡精度和速度：
- α=1.0：标准MobileNet
- α=0.75：75%通道
- α=0.5：50%通道

## 步骤1：导入必要的库

各库的作用说明：
- **numpy**：数值计算基础库，用于数组操作和数据预处理
- **tensorflow**：深度学习框架，提供模型构建、训练、推理的完整工具链
- **tensorflow.keras**：TensorFlow内置的高级API，提供简洁的模型构建接口
- **matplotlib**：数据可视化库，用于绘制训练曲线和展示图像
- **ImageDataGenerator**：图像数据增强工具，可实时生成旋转、翻转、缩放等增强样本
- **DepthwiseConv2D**：逐深度卷积层，MobileNet的核心组件，对每个通道独立做空间卷积
- **GlobalAveragePooling2D**：全局平均池化层，将每个特征图压缩为一个标量值，替代Flatten+Dense减少参数量

In [ ]:
# 载入必要的库
import numpy as np
import tensorflow as tf
# [旧写法] import keras
# [新写法] 适用于 TensorFlow >= 2.0
from tensorflow import keras

from matplotlib import pyplot as plt

# [旧写法] from keras import Model
# [旧写法] from keras.optimizers import Adam
# [旧写法] from keras.preprocessing.image import ImageDataGenerator
# [旧写法] from keras.layers import Conv2D, BatchNormalization, GlobalAveragePooling2D
# [旧写法] from keras.layers import Dense, Input, ZeroPadding2D
# [旧写法] from keras.layers import ReLU, DepthwiseConv2D
# ↑ 独立 keras 包在 TF 2.0+ 中已不推荐使用

# [新写法] 适用于 TensorFlow >= 2.0
from tensorflow.keras import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Conv2D, BatchNormalization, GlobalAveragePooling2D
from tensorflow.keras.layers import Dense, Input, ZeroPadding2D
from tensorflow.keras.layers import ReLU, DepthwiseConv2D

print(f'TensorFlow version: {tf.__version__}')

## 步骤2：定义深度可分离卷积块（代码示例6-18）

深度可分离卷积块由以下部分组成：
1. **逐深度卷积（DepthwiseConv2D）**：每个通道独立卷积
2. **批归一化（BatchNormalization）**
3. **ReLU6激活**
4. **逐点卷积（1×1 Conv2D）**：混合通道信息
5. **批归一化 + ReLU6**

**代码逐行解析**：
- `int(pointwise_conv_filters * alpha)`：将目标输出通道数乘以宽度乘子α，实现网络宽度的灵活缩放。例如 filters=128, alpha=0.75 → 实际输出96个通道
- **ZeroPadding2D的条件判断**：`strides=(1,1)` 时不需要额外填充，使用 `padding='same'`；`strides=(2,2)` 时手动在右侧和下侧各填充1像素 `((0,1),(0,1))`，配合 `padding='valid'`，确保下采样后特征图尺寸恰好减半
- **DepthwiseConv2D((3,3))**：逐深度卷积，对输入的每个通道**独立**使用一个3×3卷积核进行空间滤波，输出通道数与输入相同
- **Conv2D(filters, (1,1))**：逐点卷积（1×1卷积），在每个空间位置对所有通道做线性组合，实现**跨通道信息融合**，并改变通道数
- **use_bias=False**：不使用偏置项，因为紧随其后的BatchNormalization自带偏置参数（β），再加偏置是冗余的
- **BatchNormalization(axis=-1)**：沿通道轴做批归一化，加速收敛、稳定梯度
- **ReLU(6.)**：ReLU6激活，输出截断在[0, 6]，适合移动端量化部署

In [ ]:
#代码示例：6-18
def _depthwise_conv_block(inputs, pointwise_conv_filters, alpha,
                           strides=(1, 1), block_id=1):
    """深度可分离卷积块。

    A depthwise convolution block consists of a depthwise conv,
    batch normalization, relu6, pointwise convolution,
    batch normalization and relu6 activation.

    # Arguments

        inputs: Input tensor of shape `(rows, cols, channels)`

        pointwise_conv_filters: Integer, the dimensionality of the output space
            (i.e. the number of output filters in the pointwise convolution).

        alpha: controls the width of the network.

        strides: An integer or tuple/list of 2 integers,
            specifying the strides of the convolution
            along the width and height.

        block_id: Integer, a unique identification designating
            the block number.

    # Returns
        Output tensor of block.
    """

    pointwise_conv_filters = int(pointwise_conv_filters * alpha)

    if strides == (1, 1):
        x = inputs
    else:
        x = ZeroPadding2D(((0, 1), (0, 1)),
                          name='conv_pad_%d' % block_id)(inputs)

    # 逐深度卷积：每个通道独立使用3×3卷积核
    x = DepthwiseConv2D((3, 3),
                        padding='same' if strides == (1, 1) else 'valid',
                        strides=strides,
                        use_bias=False,
                        name='conv_dw_%d' % block_id)(x)
    x = BatchNormalization(axis=-1,
                           name='conv_dw_%d_bn' % block_id)(x)
    x = ReLU(6., name='conv_dw_%d_relu' % block_id)(x)

    # 逐点卷积：使用1×1卷积混合通道信息
    x = Conv2D(pointwise_conv_filters, (1, 1),
               padding='same',
               use_bias=False,
               strides=(1, 1),
               name='conv_pw_%d' % block_id)(x)
    x = BatchNormalization(axis=-1,
                           name='conv_pw_%d_bn' % block_id)(x)

    return ReLU(6., name='conv_pw_%d_relu' % block_id)(x)

## 步骤3：构建MobileNet模型（代码示例6-19）

### 网络整体架构

| 层 | 操作 | 输出尺寸 | 说明 |
|----|------|---------|------|
| 输入 | Input | 112×112×3 | RGB三通道图像 |
| 初始卷积 | Conv2D(32, 3×3, stride=2) | 56×56×32 | 标准卷积做初始下采样（输入通道仅3个，深度可分离优势不明显） |
| Block 1 | DW(3×3) + PW(64), stride=1 | 56×56×64 | 尺寸不变，通道数扩展到64 |
| Block 2 | DW(3×3) + PW(128), stride=2 | 28×28×128 | 下采样2倍，通道数128 |
| Block 3 | DW(3×3) + PW(256), stride=2 | 14×14×256 | 下采样2倍，通道数256 |
| Block 4 | DW(3×3) + PW(512), stride=2 | 7×7×512 | 下采样2倍，通道数512 |
| Block 5 | DW(3×3) + PW(1024), stride=2 | 4×4×1024 | 下采样2倍，通道数1024 |
| GAP | GlobalAveragePooling2D | 1024 | 每个通道取平均，4×4×1024 → 长度1024的向量 |
| 分类 | Dense(10, softmax) | 10 | 10个类别的概率输出 |

**设计要点**：
- 每次 stride=2 的下采样使特征图尺寸减半，同时通道数加倍，这是CNN的经典设计模式
- GlobalAveragePooling2D替代Flatten+Dense，零参数且防止过拟合
- 本实验为简化版MobileNet（5个block），完整版MobileNet v1有13个深度可分离卷积块

In [ ]:
#代码示例：6-19
IMSIZE = 112
alpha = 1

# 输入层
input_layer = Input([IMSIZE, IMSIZE, 3])

# 初始卷积层
x = input_layer
x = ZeroPadding2D(padding=((0, 1), (0, 1)), name='conv1_pad')(x)
x = Conv2D(32, (3, 3), padding='valid', use_bias=False, strides=(2, 2), name='conv1')(x)
x = BatchNormalization(axis=-1, name='conv1_bn')(x)
x = ReLU(6, name='conv1_relu')(x)

# 深度可分离卷积块
x = _depthwise_conv_block(x, 64, alpha, block_id=1)                        # 56×56×64
x = _depthwise_conv_block(x, 128, alpha, strides=(2, 2), block_id=2)       # 28×28×128
x = _depthwise_conv_block(x, 256, alpha, strides=(2, 2), block_id=3)       # 14×14×256
x = _depthwise_conv_block(x, 512, alpha, strides=(2, 2), block_id=4)       # 7×7×512
x = _depthwise_conv_block(x, 1024, alpha, strides=(2, 2), block_id=5)      # 4×4×1024

# 全局平均池化 + 分类层
x = GlobalAveragePooling2D()(x)
x = Dense(10, activation='softmax')(x)

model = Model(inputs=input_layer, outputs=x)
model.summary()

## 步骤4：参数量对比分析

观察上方 `model.summary()` 的输出，注意：
- 每个深度可分离卷积块的参数量远小于等效的标准卷积
- DepthwiseConv2D 层的参数量仅为 3×3×Cin = 9×Cin（例如 Cin=512 时仅 4,608 个参数）
- 逐点卷积（1×1 Conv2D）的参数量为 Cin × Cout（例如 512→1024 为 524,288 个参数）
- 总参数量相比标准卷积减少约 8-9 倍

### FLOPs（计算量）分析

参数量减少意味着模型更小，而**FLOPs减少意味着推理更快**。以Block 4（输入14×14×256，输出7×7×512）为例：

| 指标 | 标准卷积 (3×3, 256→512) | 深度可分离卷积 |
|------|------------------------|---------------|
| 参数量 | 3×3×256×512 = 1,179,648 | 3×3×256 + 256×512 = 133,376 |
| FLOPs (7×7输出) | 3²×256×512×7×7 = 57.8M | 3²×256×14×14 + 256×512×7×7 = 6.86M |
| 压缩比 | 1× | **约 8.4×** |

压缩比公式：1/Cout + 1/K² = 1/512 + 1/9 ≈ 0.113，即计算量约为标准卷积的 11.3%。

In [ ]:
# 参数量对比计算
print("=" * 60)
print("标准卷积 vs 深度可分离卷积 参数量对比")
print("=" * 60)

# 以3×3卷积，256输入通道，256输出通道为例
K = 3
C_in = 256
C_out = 256

standard_params = K * K * C_in * C_out
depthwise_params = K * K * C_in
pointwise_params = C_in * C_out
separable_params = depthwise_params + pointwise_params

print(f"\n以 {K}×{K} 卷积, {C_in} → {C_out} 通道为例:")
print(f"  标准卷积参数量:     {standard_params:>10,}")
print(f"  逐深度卷积参数量:   {depthwise_params:>10,}")
print(f"  逐点卷积参数量:     {pointwise_params:>10,}")
print(f"  深度可分离总参数量: {separable_params:>10,}")
print(f"  参数量比值:         {standard_params / separable_params:.2f}x")
print(f"  减少比例:           {(1 - separable_params / standard_params) * 100:.1f}%")

## 步骤5：编译模型（代码示例6-20）

模型编译是训练前的配置步骤，需要指定三个关键要素：

- **损失函数 `categorical_crossentropy`**：多分类交叉熵损失，公式为 L = -Σ yi·log(ŷi)。适用于互斥多分类任务（每张图片只属于一个类别），要求标签为one-hot编码格式。
- **优化器 `Adam(learning_rate=0.001)`**：Adam（Adaptive Moment Estimation）自适应学习率优化器，结合了Momentum（动量加速）和RMSProp（自适应缩放）的优点，为每个参数自动调整学习率。0.001是推荐的默认学习率。
- **评估指标 `accuracy`**：分类准确率，即预测正确的样本数占总样本数的比例，在训练和验证过程中实时监控。

In [ ]:
#代码示例：6-20
model.compile(
    loss='categorical_crossentropy',
    # [旧写法] optimizer=Adam(lr=0.001),
    # [新写法] 适用于 TensorFlow >= 2.11
    optimizer=Adam(learning_rate=0.001),
    metrics=['accuracy']
)

print("模型编译完成！")
print(f"损失函数: categorical_crossentropy")
print(f"优化器: Adam (learning_rate=0.001)")
print(f"评估指标: accuracy")

## 步骤6：训练模型（需要数据集）

以下代码展示如何加载数据并训练模型。需要准备相应的数据集目录。

### 代码结构说明

训练代码分为**数据准备**和**模型训练**两个阶段：

**1. 数据增强（ImageDataGenerator）**：
- `rescale=1./255`：将像素值从[0,255]归一化到[0,1]，加速模型收敛
- `shear_range=0.5`：随机错切变换，模拟不同拍摄角度
- `rotation_range=30`：随机旋转±30度
- `zoom_range=0.2`：随机缩放±20%
- `width_shift_range / height_shift_range=0.2`：随机水平/垂直平移±20%
- `horizontal_flip=True`：随机水平翻转
- `validation_split=0.4`：从数据集中划分40%作为验证集

**2. 数据生成器（flow_from_directory）**：
- 从指定目录自动读取图像，按子文件夹名作为类别标签
- `target_size=(IMSIZE, IMSIZE)`：将所有图像缩放到112×112
- `class_mode='categorical'`：生成one-hot编码标签，与`categorical_crossentropy`损失函数匹配

**3. 模型训练（model.fit）**：
- `steps_per_epoch=100`：每个epoch迭代100个batch（每batch 150张 → 每epoch训练15,000张）
- `epochs=5`：训练5个完整周期
- `validation_data / validation_steps`：每个epoch结束后在验证集上评估模型泛化能力

In [ ]:
# 代码示例：6-17 + 6-20（数据加载与训练）
# 注意：以下代码需要数据集，如无数据集可跳过此单元格

# import random
# random.seed(2019425)
#
# # [旧写法] from keras.preprocessing.image import ImageDataGenerator
# # [新写法] 适用于 TensorFlow >= 2.0
# from tensorflow.keras.preprocessing.image import ImageDataGenerator
#
# IMSIZE = 112
#
# datagen = ImageDataGenerator(
#     rescale=1. / 255,
#     shear_range=0.5,
#     rotation_range=30,
#     zoom_range=0.2,
#     width_shift_range=0.2,
#     height_shift_range=0.2,
#     horizontal_flip=True,
#     validation_split=0.4
# )
#
# validation_generator = datagen.flow_from_directory(
#     '/course7/data/data_mob/',
#     target_size=(IMSIZE, IMSIZE),
#     batch_size=100,
#     class_mode='categorical',
#     subset='validation'
# )
#
# train_generator = datagen.flow_from_directory(
#     '/course7/data/data_mob/',
#     target_size=(IMSIZE, IMSIZE),
#     batch_size=150,
#     class_mode='categorical',
#     subset='training'
# )
#
# # [旧写法] model.fit_generator(train_generator, steps_per_epoch=100, epochs=5, ...)
# # [新写法] 适用于 TensorFlow >= 2.1
# model.fit(
#     train_generator,
#     steps_per_epoch=100,
#     epochs=5,
#     validation_data=validation_generator,
#     validation_steps=100
# )

print("训练代码已注释，取消注释并配置数据集路径后即可运行")

## 新旧写法对照表

| 功能 | 旧写法 | 新写法 |
|------|--------|--------|
| 导入层 | `from keras.layers import DepthwiseConv2D, ReLU` | `from tensorflow.keras.layers import DepthwiseConv2D, ReLU` |
| 导入模型 | `from keras import Model` | `from tensorflow.keras import Model` |
| 学习率 | `Adam(lr=0.001)` | `Adam(learning_rate=0.001)` |
| 训练 | `model.fit_generator(...)` | `model.fit(...)` |

## 思考题

1. 深度可分离卷积为什么能大幅减少参数量？具体减少了多少倍？
2. ReLU6相比普通ReLU有什么优势？在什么场景下特别有用？
3. 宽度乘子α如何影响模型的精度和速度？
4. MobileNet适合哪些应用场景？它与ResNet相比有什么优劣？
5. 如何在精度和推理速度之间找到平衡？